# Medical Triage Classifier — LoRA Fine-Tuning

Fine-tunes `distilbert-base-uncased` with PEFT/LoRA for 3-class urgency
classification (Routine / Urgent / Emergency).

**Before running:**
1. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**
2. Upload `train.csv` and `val.csv` from your local `data/` directory
3. Run all cells in order
4. Download the `model_final/` folder when training completes

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate datasets torch scikit-learn

## 2. Upload Data

Upload `train.csv` and `val.csv` from your local `data/` directory.
Click the folder icon in the left sidebar → Upload.

In [ ]:
from google.colab import files
import os

# Upload train.csv and val.csv
print("Upload train.csv and val.csv from your local data/ directory.")
uploaded = files.upload()

# Verify uploads
for fname in ['train.csv', 'val.csv']:
    if os.path.exists(fname):
        print(f"  {fname} uploaded successfully")
    else:
        print(f"  WARNING: {fname} not found — upload it before continuing")

## 3. Verify GPU

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

## 4. Load & Tokenize Data

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# ── Constants ────────────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
LABEL_MAP = {"Routine": 0, "Urgent": 1, "Emergency": 2}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)
MAX_SEQ_LENGTH = 512

# ── Load CSVs ────────────────────────────────────────────────────────────────
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")

print(f"Train: {len(train_df)} rows")
print(f"Val:   {len(val_df)} rows")
print(f"\nTrain class distribution:")
print(train_df["urgency"].value_counts().to_string())
display(train_df.head())
display(val_df.head())

# ── Tokenize ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TriageDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

def tokenize(df):
    encodings = tokenizer(
        df["text"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )
    labels = torch.tensor(df["urgency"].map(LABEL_MAP).tolist(), dtype=torch.long)
    return TriageDataset(encodings, labels)

train_dataset = tokenize(train_df)
val_dataset = tokenize(val_df)
print(f"\nTokenization complete. Vocab size: {tokenizer.vocab_size}")

## 5. Build Model + LoRA Adapters

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

# ── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS = 5
LEARNING_RATE = 3e-4
BATCH_SIZE = 16
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

# ── Load base model ──────────────────────────────────────────────────────────
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)

# ── Apply LoRA ───────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_lin", "v_lin"],  # DistilBERT attention layers
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.to(device)
model.print_trainable_parameters()

## 6. Train

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

# ── Class weights ────────────────────────────────────────────────────────────
# Penalize mistakes on underrepresented classes more heavily to prevent
# the model from collapsing to always predicting the majority class.
train_labels = train_df["urgency"].map(LABEL_MAP).values
weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)
print(f"Class weights: { {ID_TO_LABEL[i]: round(w, 3) for i, w in enumerate(weights)} }")

class WeightedTrainer(Trainer):
    """Custom Trainer that applies class weights to cross-entropy loss."""
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ── Training args ────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f"Training on {device} for {EPOCHS} epochs...")
print(f"Hyperparams: lr={LEARNING_RATE}, batch={BATCH_SIZE}, lora_r={LORA_R}, lora_alpha={LORA_ALPHA}")
print()

train_result = trainer.train()

## 7. Evaluate

In [ ]:
eval_metrics = trainer.evaluate()

print("=== Validation Results ===")
print(f"Accuracy:  {eval_metrics['eval_accuracy']:.4f}")
print(f"F1 Macro:  {eval_metrics['eval_f1_macro']:.4f}")
print(f"Loss:      {eval_metrics['eval_loss']:.4f}")
print(f"\nTrain loss: {train_result.training_loss:.4f}")
print(f"Train time: {train_result.metrics.get('train_runtime', 0):.1f}s")

## 8. Save Model

In [ ]:
# Save the fine-tuned LoRA model + tokenizer
SAVE_DIR = "model_final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# List saved files
print(f"Model saved to {SAVE_DIR}/")
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:40s} {size / 1024:.1f} KB")

## 9. Quick Smoke Test

In [ ]:
# Reload and test the saved model
from peft import PeftModel

test_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
test_base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, id2label=ID_TO_LABEL, label2id=LABEL_MAP,
)
test_model = PeftModel.from_pretrained(test_base, SAVE_DIR)
test_model.to(device)
test_model.eval()

samples = [
    ("Patient presents with acute chest pain radiating to left arm. Diaphoretic, BP 180/110.", "Emergency"),
    ("45-year-old female for annual wellness exam. No acute complaints. Well-controlled HTN.", "Routine"),
    ("Worsening abdominal pain x3 days, RLQ, low-grade fever 100.4F. Rebound tenderness.", "Urgent"),
]

print("=== Smoke Test ===")
for text, expected in samples:
    inputs = test_tokenizer(text, truncation=True, padding="max_length",
                            max_length=MAX_SEQ_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = test_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).squeeze()
    pred_id = torch.argmax(probs).item()
    pred_label = ID_TO_LABEL[pred_id]
    confidence = probs[pred_id].item()
    match = "OK" if pred_label == expected else "MISMATCH"
    print(f"  [{match}] Expected: {expected:10s} | Predicted: {pred_label:10s} ({confidence:.1%})")
    print(f"         Scores: {', '.join(f'{ID_TO_LABEL[i]}: {probs[i].item():.1%}' for i in range(NUM_LABELS))}")

## 10. Download Model

Downloads `model_final/` as a zip file. Unzip it into your local
`model_artifacts/final/` directory, then run:

```bash
python evaluator.py
streamlit run app.py
```

In [ ]:
import shutil

# Zip the model directory
shutil.make_archive("model_final", "zip", ".", SAVE_DIR)
print(f"Created model_final.zip ({os.path.getsize('model_final.zip') / 1024 / 1024:.1f} MB)")

# Download
files.download("model_final.zip")
print("\nDownload started. After unzipping, place contents in model_artifacts/final/")